In [0]:
# ---------------------------------------------------------------------------
# 1. Configuration & Dataset Registry (Metadata-Driven Ingestion)
# ---------------------------------------------------------------------------
from typing import Any, Dict, List
from pyspark.sql import functions as F

CATALOG_NAME = "aircargo_dev"
BRONZE_SCHEMA = "bronze"

LANDING_BASE_PATH = "/Volumes/aircargo_dev/landing/raw_files"
# Keep Auto Loader checkpoints & inferred schemas separate from source data folders
AUTOLOADER_STATE_PATH = f"{LANDING_BASE_PATH}/_autoloader_state"

DATASET_CONFIGS: List[Dict[str, Any]] = [
    {
        "dataset_name": "airports",
        "file_format": "csv",
        "source_path": f"{LANDING_BASE_PATH}/airports",
        "target_table": f"{CATALOG_NAME}.{BRONZE_SCHEMA}.airports_raw",
        "reader_options": {
            "header": "true",
            "cloudFiles.inferColumnTypes": "true",
        },
    },
    {
        "dataset_name": "weather",
        "file_format": "json",
        "source_path": f"{LANDING_BASE_PATH}/weather",
        "target_table": f"{CATALOG_NAME}.{BRONZE_SCHEMA}.weather_raw",
        "reader_options": {
            "cloudFiles.inferColumnTypes": "true",
        },
    },
    {
        "dataset_name": "cargo_manifests",
        "file_format": "json",
        "source_path": f"{LANDING_BASE_PATH}/cargo_manifests",
        "target_table": f"{CATALOG_NAME}.{BRONZE_SCHEMA}.cargo_manifests_raw",
        "reader_options": {
            "cloudFiles.inferColumnTypes": "true",
        },
    },
]

print(f"Configured {len(DATASET_CONFIGS)} batch datasets for Auto Loader ingestion.")

In [0]:
# ---------------------------------------------------------------------------
# 2. Ensure Target Bronze Schema Exists (Idempotent Guard)
# ---------------------------------------------------------------------------
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG_NAME}.{BRONZE_SCHEMA}")
print(f"Verified schema: {CATALOG_NAME}.{BRONZE_SCHEMA}")

In [0]:
# ---------------------------------------------------------------------------
# 3. Reusable Auto Loader Ingestion Function
# ---------------------------------------------------------------------------
def ingest_dataset_to_bronze(config: Dict[str, Any]) -> Dict[str, Any]:
    """Incrementally ingest files from a Volume folder into a Bronze Delta table."""
    dataset_name = config["dataset_name"]
    target_table = config["target_table"]
    checkpoint_path = f"{AUTOLOADER_STATE_PATH}/{dataset_name}/checkpoint"
    schema_path = f"{AUTOLOADER_STATE_PATH}/{dataset_name}/schema"

    # Capture current Delta table version before streaming run
    version_before = (
        spark.sql(f"DESCRIBE HISTORY {target_table} LIMIT 1").first()["version"]
        if spark.catalog.tableExists(target_table)
        else -1
    )

    cloud_files_options = {
        "cloudFiles.format": config["file_format"],
        "cloudFiles.schemaLocation": schema_path,
        "cloudFiles.schemaEvolutionMode": "addNewColumns",
        "cloudFiles.rescuedDataColumn": "_rescued_data",
        **config.get("reader_options", {}),
    }

    raw_stream_df = (
        spark.readStream
        .format("cloudFiles")
        .options(**cloud_files_options)
        .load(config["source_path"])
    )

    enriched_df = (
        raw_stream_df
        .withColumn("_ingest_timestamp", F.current_timestamp())
        .withColumn("_source_file", F.col("_metadata.file_path"))
        .withColumn("_source_file_mod_time", F.col("_metadata.file_modification_time"))
        .withColumn("_source_system", F.lit(f"landing_volume_{dataset_name}"))
    )

    query = (
        enriched_df.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", checkpoint_path)
        .option("mergeSchema", "true")
        .trigger(availableNow=True)
        .toTable(target_table)
    )
    query.awaitTermination()

    # Sum exact rows committed to Delta across all micro-batches in this run
    new_commits = (
        spark.sql(f"DESCRIBE HISTORY {target_table}")
        .filter(F.col("version") > version_before)
        .select("operationMetrics")
        .collect()
    )
    rows_processed = sum(
        int((row["operationMetrics"] or {}).get("numOutputRows", 0))
        for row in new_commits
    )

    print(f"[{dataset_name}] -> {target_table} | new rows committed: {rows_processed}")
    return {
        "dataset_name": dataset_name,
        "target_table": target_table,
        "rows_processed": rows_processed,
    }

In [0]:
# ---------------------------------------------------------------------------
# 4. Execute Incremental Ingestion for All Reference Datasets
# ---------------------------------------------------------------------------
run_summary = []
for dataset_cfg in DATASET_CONFIGS:
    result = ingest_dataset_to_bronze(dataset_cfg)
    run_summary.append(result)

print("\nAuto Loader batch run finished.")

In [0]:
# ---------------------------------------------------------------------------
# 5. Reconciliation & Data Quality Check (_rescued_data Audit)
# ---------------------------------------------------------------------------
audit_queries = [
    f"""
    SELECT
        '{cfg["dataset_name"]}' AS dataset_name,
        '{cfg["target_table"]}' AS target_table,
        COUNT(*) AS total_rows,
        COUNT(CASE WHEN _rescued_data IS NOT NULL THEN 1 END) AS rescued_rows,
        COUNT(DISTINCT _source_file) AS distinct_source_files,
        MAX(_ingest_timestamp) AS last_ingest_ts
    FROM {cfg["target_table"]}
    """
    for cfg in DATASET_CONFIGS
]

display(spark.sql(" UNION ALL ".join(audit_queries)))
display(spark.table(f"{CATALOG_NAME}.{BRONZE_SCHEMA}.cargo_manifests_raw").limit(5))